# Phase 2B — Controlled Huber Loss Experiment: δ=1 vs δ=2

Evaluation on **VALIDATION SET ONLY** (Test set remains strictly frozen).

**Research Question:**
Does increasing Huber loss transition threshold from $\delta=1.0$ to $\delta=2.0$ original magnitude units reduce upper-tail shrinkage and improve validation performance/prediction variability in catalog BiLSTM magnitude regression?

**Protocol & Settings (Fixed):**
- Architecture: 2-layer BiLSTM, hidden_size=128, dropout=0.3144753602375439, Linear(256 -> 1)
- Optimizer: AdamW, learning_rate=0.0004549047669323766, weight_decay=0.00012155233771053111, gradient_clip=1.0
- Sampling: Weighted sampling Variant C (regression relevance complete-window sampling, seed 42)
- Training: Maximum 50 epochs, early stopping patience 8 on validation MAE, batch_size=128
- Seed: 42, deterministic PyTorch CUDA algorithms
- Target: Next retained ML event magnitude (unscaled identity transform)
- Output Directory: `phase2_bilstm_results/huber_experiment/`


In [1]:
from pathlib import Path
import os, sys, json, hashlib, time, random, pickle, ast, gc
from datetime import datetime, timezone

ROOT = Path.cwd().resolve()
if ROOT.name != "bilstm new updated":
    ROOT = ROOT / "bilstm new updated"
sys.path.insert(0, str(ROOT / ".training-runtime"))
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import numpy as np
import pandas as pd
import torch
from torch import nn
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

DATA = ROOT / "dataset" / "v2_timestamp_fixed"
AUG = ROOT / "augmentation_experiment" / "v2_timestamp_fixed"
OUT = ROOT / "phase2_bilstm_results" / "huber_experiment"
for folder in [OUT, OUT / "plots"]:
    folder.mkdir(parents=True, exist_ok=True)

SEED = 42; MAX_EPOCHS = 50; PATIENCE = 8; HUBER_DELTA = 2.0; CLIP = 1.0
torch.set_num_threads(4)
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = False; torch.backends.cudnn.allow_tf32 = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def seed_all():
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

def sha(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for b in iter(lambda: f.read(1024 * 1024), b""):
            h.update(b)
    return h.hexdigest()

def content_sha(a):
    if isinstance(a, torch.Tensor): a = a.detach().cpu().contiguous().numpy()
    a = np.ascontiguousarray(a)
    h = hashlib.sha256(); h.update(str(a.dtype).encode()); h.update(str(a.shape).encode()); h.update(a.tobytes())
    return h.hexdigest()

def replace_with_retry(tmp, path):
    for attempt in range(30):
        try:
            tmp.replace(path); return
        except PermissionError:
            if attempt == 29: raise
            time.sleep(min(.1 * (attempt + 1), .5))

def save_json(path, value):
    path = Path(path); tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(value, indent=2, allow_nan=False), encoding="utf-8")
    replace_with_retry(tmp, path)

def save_torch(path, value):
    path = Path(path); tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(value, tmp); replace_with_retry(tmp, path)

# Verify target definition and units
target_definition = json.loads((DATA / "target_definition.json").read_text())
scaling = json.loads((DATA / "catalog_train_scalers.json").read_text())
aug_config = json.loads((AUG / "augmentation_config.json").read_text())
dataset_hash = sha(DATA / "bilstm_catalog_seq20.pt")

# Explicit assertion: Target is in UNTRANSFORMED, UNCONVERTED ML MAGNITUDE UNITS
assert target_definition["target_transform"] == {"name": "identity", "offset": 0.0, "scale": 1.0}
print("[UNIT VERIFICATION] target_transform is identity (offset=0.0, scale=1.0).")
print(f"[UNIT VERIFICATION] Delta is measured in ORIGINAL magnitude units (ML).")
print(f"[UNIT VERIFICATION] Existing model used Huber delta=1.0 ML. New model uses Huber delta={HUBER_DELTA} ML.")

# Load training and validation tensors ONLY. Test set is strictly avoided.
mapped = torch.load(DATA / "bilstm_catalog_seq20.pt", map_location="cpu", weights_only=True, mmap=True)
train_X = mapped["X_train"].clone(); train_y = mapped["y_reg_train"].clone()
train_binary = mapped["y_binary_train"].clone()
val_X = mapped["X_val"].clone(); val_y = mapped["y_reg_val"].clone()
del mapped

assert tuple(train_X.shape) == (4973, 20, 10) and tuple(val_X.shape) == (1065, 20, 10)
assert all(torch.isfinite(t).all() for t in [train_X, train_y, val_X, val_y])
print(f"[DATA LOAD] Train: {train_X.shape}, Val: {val_X.shape} (Test data strictly unread).")

# Prepare Variant C data consumer using approved preprocessing functions
parent_manifest = pd.read_csv(DATA / "split_manifest.csv", usecols=["split", "sample_index", "input_event_ids", "target_event_id"])
train_manifest = parent_manifest.loc[parent_manifest.split.eq("train")].reset_index(drop=True)
input_ids = np.array([json.loads(v) for v in train_manifest.input_event_ids])
unique_ids = np.unique(input_ids); id_rows = {v: i for i, v in enumerate(unique_ids)}
history_rows = np.array([[id_rows[v] for v in row] for row in input_ids])

aug_notebook = json.loads((ROOT / "phase1b_imbalance_noise_preprocessing.ipynb").read_text())
function_names = {"draw_parent_indices", "coordinate_noise", "diagnostic_variant", "prepare_training_epoch"}
function_nodes = []
for cell in aug_notebook["cells"]:
    if cell["cell_type"] == "code":
        function_nodes.extend(node for node in ast.parse("".join(cell["source"])).body
                              if isinstance(node, ast.FunctionDef) and node.name in function_names)
assert {node.name for node in function_nodes} == function_names
consumer_source = ast.unparse(ast.Module(body=function_nodes, type_ignores=[]))
consumer_hash = hashlib.sha256(consumer_source.encode()).hexdigest()

with np.load(AUG / "sampling_plan.npz", allow_pickle=False) as plan:
    approved_weights = plan["weights"].copy()

scope = {"np": np, "CONFIG": aug_config, "SEED": SEED, "N": len(train_y),
         "probabilities": approved_weights / approved_weights.sum(),
         "X": train_X.numpy(), "y": train_y.numpy(), "binary": train_binary.numpy(),
         "unique_history_ids": unique_ids, "history_rows": history_rows,
         "mean": np.array(scaling["scaler"]["mean"]), "scale": np.array(scaling["scaler"]["scale"]), "source_ok": True}
exec(compile(ast.Module(body=function_nodes, type_ignores=[]), "<approved-v2-augmentation-consumers>", "exec"), scope)
epoch_data = scope["prepare_training_epoch"]
parents, x, y, b = epoch_data("C", 0)
assert np.array_equal(y, train_y.numpy()[parents])
del parents, x, y, b

train_gpu = train_X.to(DEVICE); val_gpu = val_X.to(DEVICE)
print("[SETUP COMPLETE] Device:", DEVICE, "Approved Variant C generator verified.")


[UNIT VERIFICATION] target_transform is identity (offset=0.0, scale=1.0).
[UNIT VERIFICATION] Delta is measured in ORIGINAL magnitude units (ML).
[UNIT VERIFICATION] Existing model used Huber delta=1.0 ML. New model uses Huber delta=2.0 ML.
[DATA LOAD] Train: torch.Size([4973, 20, 10]), Val: torch.Size([1065, 20, 10]) (Test data strictly unread).
[SETUP COMPLETE] Device: cuda Approved Variant C generator verified.


## Model Architecture & Loss Implementation

The architecture is identical to Phase 2: a 2-layer Bidirectional LSTM with hidden size 128, concatenated forward/backward terminal hidden states (`h_n[-2]` and `h_n[-1]`), dropout, and a linear head producing a single scalar output $M_L$.

The loss function is `nn.HuberLoss(delta=2.0)`.


In [3]:
class BiLSTMRegressor(nn.Module):
    def __init__(self, hidden_size, layers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(10, hidden_size, num_layers=layers, batch_first=True, bidirectional=True,
                            dropout=dropout if layers > 1 else 0.)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(2 * hidden_size, 1)
    def forward(self, x):
        _, (h, _) = self.lstm(x)
        return self.head(self.dropout(torch.cat([h[-2], h[-1]], dim=1))).squeeze(-1)

def model_config(cfg):
    return {k: cfg[k] for k in ["hidden_size", "layers", "dropout"]}

def state_hash(model):
    h = hashlib.sha256()
    for name, tensor in model.state_dict().items():
        h.update(name.encode()); h.update(tensor.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()

def sync():
    if DEVICE.type == "cuda": torch.cuda.synchronize()

def predict(model, x, batch=256):
    model.eval(); out = []
    with torch.inference_mode():
        for start in range(0, len(x), batch):
            out.append(model(x[start:start+batch].to(DEVICE)).cpu())
    return torch.cat(out).numpy()

def metric_values(actual, pred, delta=HUBER_DELTA):
    actual = np.asarray(actual, dtype=np.float64); pred = np.asarray(pred, dtype=np.float64)
    residual = pred - actual; absolute = np.abs(residual); high = actual >= 4.5
    return {
        "mae": float(absolute.mean()),
        "rmse": float(np.sqrt(np.mean(residual**2))),
        "r2": float(1 - np.sum(residual**2) / np.sum((actual - actual.mean())**2)),
        "huber": float(np.where(absolute <= delta, .5 * residual**2, delta * (absolute - .5 * delta)).mean()),
        "pred_mean": float(pred.mean()),
        "pred_std": float(pred.std()),
        "pred_min": float(pred.min()),
        "pred_max": float(pred.max()),
        "high_count": int(high.sum()),
        "high_mae": float(absolute[high].mean()) if high.any() else None,
        "high_underprediction_rate": float((residual[high] < 0).mean()) if high.any() else None,
        "high_bias": float(residual[high].mean()) if high.any() else None
    }

# Load best hyperparameters from Phase 2 Optuna tuning
best_hp_path = ROOT / "phase2_bilstm_results" / "best_hyperparameters.json"
best_cfg = json.loads(best_hp_path.read_text())["parameters"]
print("[HYPERPARAMETERS LOADED]:", best_cfg)


[HYPERPARAMETERS LOADED]: {'hidden_size': 128, 'layers': 2, 'dropout': 0.3144753602375439, 'learning_rate': 0.0004549047669323766, 'weight_decay': 0.00012155233771053111, 'batch_size': 128}


## Training BiLSTM with Huber δ=2.0 on Variant C


In [5]:
folder = OUT
best_path = folder / "best.pt"
last_path = folder / "last.pt"
summary_path = folder / "fit_summary.json"

seed_all()
model = BiLSTMRegressor(**model_config(best_cfg)).to(DEVICE)
initial_hash = state_hash(model)
optimizer = torch.optim.AdamW(model.parameters(), lr=best_cfg["learning_rate"], weight_decay=best_cfg["weight_decay"])
loss_fn = nn.HuberLoss(delta=HUBER_DELTA)

history = []; best_mae = float("inf"); best_epoch = 0; stale = 0; elapsed = 0.
parameter_count = sum(p.numel() for p in model.parameters())

print(f"[TRAINING START] BiLSTM Huber delta={HUBER_DELTA}, Parameters: {parameter_count:,}")
for epoch in range(1, MAX_EPOCHS + 1):
    if stale >= PATIENCE: break
    sync(); epoch_start = time.perf_counter()
    parents, x_epoch, y_epoch, _ = epoch_data("C", epoch - 1)
    assert np.array_equal(y_epoch, train_y.numpy()[parents])
    x_device = torch.from_numpy(x_epoch).to(DEVICE); y_device = torch.from_numpy(y_epoch).to(DEVICE)
    
    model.train(); loss_sum = torch.zeros((), device=DEVICE)
    for start in range(0, len(parents), best_cfg["batch_size"]):
        end = min(start + best_cfg["batch_size"], len(parents))
        optimizer.zero_grad(set_to_none=True)
        predictions = model(x_device[start:end]); loss = loss_fn(predictions, y_device[start:end])
        loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), CLIP); optimizer.step()
        loss_sum += loss.detach() * (end - start)
    
    epoch_loss = float(loss_sum.item() / len(parents))
    train_pred = predict(model, train_gpu); val_pred = predict(model, val_gpu)
    train_metrics = metric_values(train_y.numpy(), train_pred, delta=HUBER_DELTA)
    val_metrics = metric_values(val_y.numpy(), val_pred, delta=HUBER_DELTA)
    
    improved = val_metrics["mae"] < best_mae
    if improved:
        best_mae = val_metrics["mae"]; best_epoch = epoch; stale = 0
        save_torch(best_path, {
            "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
            "model_config": model_config(best_cfg), "hyperparameters": best_cfg,
            "dataset_version": "v2_timestamp_fixed", "dataset_sha256": dataset_hash,
            "augmentation_variant": "C", "target_definition": target_definition,
            "best_epoch": epoch, "validation_mae": best_mae, "seed": SEED,
            "parameter_count": parameter_count, "initialization_sha256": initial_hash,
            "optimizer": "AdamW", "loss": f"Huber_delta{HUBER_DELTA}"
        })
    else:
        stale += 1
    
    sync(); elapsed += time.perf_counter() - epoch_start
    history.append({
        "epoch": epoch, "training_augmented_huber": epoch_loss,
        **{"train_" + k: v for k, v in train_metrics.items()},
        **{"val_" + k: v for k, v in val_metrics.items()},
        "unique_parents": int(len(np.unique(parents))),
        "repeated_draws": int(len(parents) - len(np.unique(parents))),
        "parent_indices_sha256": content_sha(parents),
        "effective_high_fraction": float((y_epoch >= 4.5).mean()),
        "elapsed_seconds": elapsed
    })
    save_json(folder / "history.json", history)
    save_torch(last_path, {
        "model_state": model.state_dict(), "optimizer_state": optimizer.state_dict(),
        "history": history, "epoch": epoch, "best_mae": best_mae, "best_epoch": best_epoch,
        "stale": stale, "elapsed_seconds": elapsed, "initialization_sha256": initial_hash
    })
    print(f"Epoch {epoch:2d}: Val MAE={val_metrics['mae']:.4f}, Val RMSE={val_metrics['rmse']:.4f}, "
          f"Val Pred Std={val_metrics['pred_std']:.4f}, High MAE={val_metrics['high_mae']:.4f} "
          f"{'[BEST]' if improved else ''}")
    del x_device, y_device

summary = {
    "status": "COMPLETE", "best_epoch": best_epoch, "best_validation_mae": best_mae,
    "epochs_run": len(history), "training_seconds": elapsed, "parameter_count": parameter_count,
    "initialization_sha256": initial_hash, "best_checkpoint_sha256": sha(best_path),
    "huber_delta": HUBER_DELTA
}
save_json(summary_path, summary)
print("[TRAINING COMPLETE] Best Epoch:", best_epoch, "Best Val MAE:", f"{best_mae:.4f}")


[TRAINING START] BiLSTM Huber delta=2.0, Parameters: 538,881
Epoch  1: Val MAE=1.3714, Val RMSE=1.8699, Val Pred Std=1.3215, High MAE=2.3179 [BEST]
Epoch  2: Val MAE=0.9135, Val RMSE=1.2491, Val Pred Std=0.6999, High MAE=2.1692 [BEST]
Epoch  3: Val MAE=0.6862, Val RMSE=0.8922, Val Pred Std=0.3097, High MAE=1.9250 [BEST]
Epoch  4: Val MAE=0.6136, Val RMSE=0.7974, Val Pred Std=0.1824, High MAE=1.8436 [BEST]
Epoch  5: Val MAE=0.5684, Val RMSE=0.7446, Val Pred Std=0.1160, High MAE=1.7750 [BEST]
Epoch  6: Val MAE=0.5315, Val RMSE=0.6996, Val Pred Std=0.1226, High MAE=1.6919 [BEST]
Epoch  7: Val MAE=0.5160, Val RMSE=0.6848, Val Pred Std=0.1062, High MAE=1.6643 [BEST]
Epoch  8: Val MAE=0.5338, Val RMSE=0.7059, Val Pred Std=0.1013, High MAE=1.7217 
Epoch  9: Val MAE=0.4797, Val RMSE=0.6346, Val Pred Std=0.1617, High MAE=1.4967 [BEST]
Epoch 10: Val MAE=0.4962, Val RMSE=0.6577, Val Pred Std=0.1078, High MAE=1.5858 
Epoch 11: Val MAE=0.4698, Val RMSE=0.6243, Val Pred Std=0.1248, High MAE=1.4658 [

## Validation Evaluation and Baseline Comparison (δ=1 vs δ=2)

We evaluate the best checkpoint on validation data and compare directly with the baseline Variant C ($\delta=1.0$) model.


In [7]:
# Reload best checkpoint
ck = torch.load(best_path, map_location="cpu", weights_only=True)
best_model = BiLSTMRegressor(**ck["model_config"]).to(DEVICE)
best_model.load_state_dict(ck["model_state"])
best_model.eval()

val_pred = predict(best_model, val_gpu)
train_pred = predict(best_model, train_gpu)

vm_delta2 = metric_values(val_y.numpy(), val_pred, delta=HUBER_DELTA)
tm_delta2 = metric_values(train_y.numpy(), train_pred, delta=HUBER_DELTA)

# Save validation predictions
np.savez_compressed(folder / "validation_predictions.npz", actual=val_y.numpy(), prediction=val_pred)

# Load baseline Variant C (delta=1) validation predictions and metrics
base_pred_data = np.load(ROOT / "phase2_bilstm_results" / "variants" / "C" / "validation_predictions.npz")
val_pred_delta1 = base_pred_data["prediction"]
vm_delta1 = metric_values(val_y.numpy(), val_pred_delta1, delta=1.0)

comp_df = pd.DataFrame([
    {
        "Metric": "Huber Delta (ML units)",
        "Baseline (delta=1)": "1.0",
        "Experiment (delta=2)": "2.0",
        "Difference (delta=2 minus delta=1)": "+1.0"
    },
    {
        "Metric": "Validation MAE",
        "Baseline (delta=1)": f"{vm_delta1['mae']:.4f}",
        "Experiment (delta=2)": f"{vm_delta2['mae']:.4f}",
        "Difference (delta=2 minus delta=1)": f"{vm_delta2['mae'] - vm_delta1['mae']:+.4f}"
    },
    {
        "Metric": "Validation RMSE",
        "Baseline (delta=1)": f"{vm_delta1['rmse']:.4f}",
        "Experiment (delta=2)": f"{vm_delta2['rmse']:.4f}",
        "Difference (delta=2 minus delta=1)": f"{vm_delta2['rmse'] - vm_delta1['rmse']:+.4f}"
    },
    {
        "Metric": "Validation R2",
        "Baseline (delta=1)": f"{vm_delta1['r2']:.4f}",
        "Experiment (delta=2)": f"{vm_delta2['r2']:.4f}",
        "Difference (delta=2 minus delta=1)": f"{vm_delta2['r2'] - vm_delta1['r2']:+.4f}"
    },
    {
        "Metric": "ML>=4.5 Validation MAE",
        "Baseline (delta=1)": f"{vm_delta1['high_mae']:.4f}",
        "Experiment (delta=2)": f"{vm_delta2['high_mae']:.4f}",
        "Difference (delta=2 minus delta=1)": f"{vm_delta2['high_mae'] - vm_delta1['high_mae']:+.4f}"
    },
    {
        "Metric": "ML>=4.5 Underprediction Rate",
        "Baseline (delta=1)": f"{vm_delta1['high_underprediction_rate']*100:.1f}%",
        "Experiment (delta=2)": f"{vm_delta2['high_underprediction_rate']*100:.1f}%",
        "Difference (delta=2 minus delta=1)": f"{(vm_delta2['high_underprediction_rate'] - vm_delta1['high_underprediction_rate'])*100:+.1f}%"
    },
    {
        "Metric": "ML>=4.5 Signed Bias",
        "Baseline (delta=1)": f"{vm_delta1['high_bias']:.4f}",
        "Experiment (delta=2)": f"{vm_delta2['high_bias']:.4f}",
        "Difference (delta=2 minus delta=1)": f"{vm_delta2['high_bias'] - vm_delta1['high_bias']:+.4f}"
    },
    {
        "Metric": "Prediction Mean",
        "Baseline (delta=1)": f"{vm_delta1['pred_mean']:.4f}",
        "Experiment (delta=2)": f"{vm_delta2['pred_mean']:.4f}",
        "Difference (delta=2 minus delta=1)": f"{vm_delta2['pred_mean'] - vm_delta1['pred_mean']:+.4f}"
    },
    {
        "Metric": "Prediction Std Dev",
        "Baseline (delta=1)": f"{vm_delta1['pred_std']:.4f}",
        "Experiment (delta=2)": f"{vm_delta2['pred_std']:.4f}",
        "Difference (delta=2 minus delta=1)": f"{vm_delta2['pred_std'] - vm_delta1['pred_std']:+.4f}"
    },
    {
        "Metric": "Prediction Range [Min, Max]",
        "Baseline (delta=1)": f"[{vm_delta1['pred_min']:.3f}, {vm_delta1['pred_max']:.3f}]",
        "Experiment (delta=2)": f"[{vm_delta2['pred_min']:.3f}, {vm_delta2['pred_max']:.3f}]",
        "Difference (delta=2 minus delta=1)": f"Span: {vm_delta2['pred_max']-vm_delta2['pred_min']:.3f} vs {vm_delta1['pred_max']-vm_delta1['pred_min']:.3f}"
    },
    {
        "Metric": "Clean Train MAE",
        "Baseline (delta=1)": "0.4309",
        "Experiment (delta=2)": f"{tm_delta2['mae']:.4f}",
        "Difference (delta=2 minus delta=1)": f"{tm_delta2['mae'] - 0.4309:+.4f}"
    },
    {
        "Metric": "Generalization Gap (Val - Train MAE)",
        "Baseline (delta=1)": "+0.0415",
        "Experiment (delta=2)": f"{vm_delta2['mae'] - tm_delta2['mae']:+.4f}",
        "Difference (delta=2 minus delta=1)": f"{(vm_delta2['mae'] - tm_delta2['mae']) - 0.0415:+.4f}"
    },
    {
        "Metric": "Best Epoch / Total Epochs",
        "Baseline (delta=1)": "12 / 20",
        "Experiment (delta=2)": f"{summary['best_epoch']} / {summary['epochs_run']}",
        "Difference (delta=2 minus delta=1)": "-"
    }
])

print("=" * 75)
print("CONTROLLED HUBER LOSS EXPERIMENT: VALIDATION SET RESULTS (DELTA=1 vs DELTA=2)")
print("=" * 75)
print(comp_df.to_string(index=False))
comp_df.to_csv(folder / "huber_loss_comparison.csv", index=False)

# Save detailed JSON metrics
metrics_payload = {
    "experiment": "Huber_delta2_vs_delta1",
    "delta1_baseline": vm_delta1,
    "delta2_experiment": vm_delta2,
    "delta2_clean_train": tm_delta2,
    "generalization_gap_mae": float(vm_delta2['mae'] - tm_delta2['mae']),
    "fit_summary": summary
}
save_json(folder / "huber_delta2_metrics.json", metrics_payload)


CONTROLLED HUBER LOSS EXPERIMENT: VALIDATION SET RESULTS (DELTA=1 vs DELTA=2)
                              Metric Baseline (delta=1) Experiment (delta=2) Difference (delta=2 minus delta=1)
              Huber Delta (ML units)                1.0                  2.0                               +1.0
                      Validation MAE             0.4724               0.4698                            -0.0026
                     Validation RMSE             0.6242               0.6243                            +0.0002
                       Validation R2             0.0162               0.0158                            -0.0005
              ML>=4.5 Validation MAE             1.4214               1.4658                            +0.0443
        ML>=4.5 Underprediction Rate             100.0%               100.0%                              +0.0%
                 ML>=4.5 Signed Bias            -1.4214              -1.4658                            -0.0443
                     Predi

## Diagnostic Figures: Learning Curves & Prediction Distributions


In [9]:
# 1. Learning curves
base_history = json.loads((ROOT / "phase2_bilstm_results" / "variants" / "C" / "history.json").read_text())

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot([h["epoch"] for h in base_history], [h["val_mae"] for h in base_history],
             label="Baseline (Huber $\\delta=1.0$)", color="steelblue", marker="o", markersize=4)
axes[0].plot([h["epoch"] for h in history], [h["val_mae"] for h in history],
             label="Experiment (Huber $\\delta=2.0$)", color="crimson", marker="s", markersize=4)
axes[0].set_title("Validation MAE Across Epochs", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Validation MAE (ML)")
axes[0].grid(True, alpha=0.3); axes[0].legend()

axes[1].plot([h["epoch"] for h in base_history], [h["val_pred_std"] for h in base_history if "val_pred_std" in h] or [h["val_rmse"] for h in base_history],
             label="Baseline Val RMSE ($\\delta=1.0$)", color="steelblue", linestyle="--")
axes[1].plot([h["epoch"] for h in history], [h["val_rmse"] for h in history],
             label="Experiment Val RMSE ($\\delta=2.0$)", color="crimson", linestyle="-")
axes[1].set_title("Validation RMSE Across Epochs", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Validation RMSE (ML)")
axes[1].grid(True, alpha=0.3); axes[1].legend()

plt.tight_layout()
plt.savefig(folder / "plots" / "training_validation_curves.png", dpi=150)
plt.close()

# 2. Prediction distribution vs Actual Targets
fig, ax = plt.subplots(figsize=(8, 5))
bins = np.linspace(1.5, 7.5, 35)
ax.hist(val_y.numpy(), bins=bins, alpha=0.35, color="gray", label=f"Actual Validation Targets (std={val_y.numpy().std():.2f})", density=True)
ax.hist(val_pred_delta1, bins=bins, alpha=0.6, color="steelblue", label=f"Baseline $\\delta=1.0$ Preds (std={val_pred_delta1.std():.2f})", density=True)
ax.hist(val_pred, bins=bins, alpha=0.6, color="crimson", label=f"Experiment $\\delta=2.0$ Preds (std={val_pred.std():.2f})", density=True)
ax.set_title("Validation Target vs Predicted Magnitude Distribution", fontsize=12, fontweight="bold")
ax.set_xlabel("Magnitude (ML)"); ax.set_ylabel("Density")
ax.grid(True, alpha=0.3); ax.legend()

plt.tight_layout()
plt.savefig(folder / "plots" / "prediction_distribution_comparison.png", dpi=150)
plt.close()
print("[PLOTS SAVED] Diagnostic figures saved to:", folder / "plots")


[PLOTS SAVED] Diagnostic figures saved to: C:\Users\choud\Desktop\Minor project\bilstm new updated\phase2_bilstm_results\huber_experiment\plots
